In [2]:
import rasterio
from rasterio.vrt import WarpedVRT
from rasterio.warp import Resampling
import numpy as np

# --- File paths ---
r_compound = r"D:\Phd Research\Final_Raster\100yr_compound_flood_base_stat.tif"
r_bathtub  = r"D:\Phd Research\Final_Raster\Bathtub_depth_100yr_surge_SLR.tif"
r_dem      = r"D:\Phd Research\Final_Raster\land_part_area_DEM.tif"

# --- Depth range (m) ---
MIN_DEPTH, MAX_DEPTH = 2.5, 6.0

def elev_stats_for_depth_band(depth_path, dem_path, dmin=2.5, dmax=6.0):
    """Reproject DEM to match depth raster grid, then report DEM elevation stats
       for cells where depth ∈ [dmin, dmax]."""
    with rasterio.open(depth_path) as dsrc, rasterio.open(dem_path) as dem_src:
        # Read depth as masked array
        depth = dsrc.read(1, masked=True)
        # Build a VRT that warps DEM to match depth raster grid
        vrt_opts = dict(
            crs=dsrc.crs,
            transform=dsrc.transform,
            height=dsrc.height,
            width=dsrc.width,
            resampling=Resampling.bilinear  # smooth + appropriate for elevation
        )
        with WarpedVRT(dem_src, **vrt_opts) as dem_vrt:
            dem = dem_vrt.read(1, masked=True)

    # Valid where both are valid and depth is positive
    valid = (~depth.mask) & (~dem.mask) & np.isfinite(depth) & np.isfinite(dem)
    band = valid & (depth >= dmin) & (depth <= dmax)

    if band.sum() == 0:
        return None  # no pixels in this depth band

    z = dem[band].compressed()
    return {
        "elev_min_m": float(np.min(z)),
        "elev_max_m": float(np.max(z)),
        "elev_mean_m": float(np.mean(z)),
        "n_pixels": int(z.size)
    }

# --- Run for both rasters ---
for label, depth_path in [("Compound", r_compound), ("Bathtub", r_bathtub)]:
    out = elev_stats_for_depth_band(depth_path, r_dem, MIN_DEPTH, MAX_DEPTH)
    if out is None:
        print(f"{label}: No cells with depth between {MIN_DEPTH}–{MAX_DEPTH} m.")
    else:
        print(f"{label} (depth {MIN_DEPTH}–{MAX_DEPTH} m):")
        print(f"  Elevation range: {out['elev_min_m']:.2f} – {out['elev_max_m']:.2f} m")
        print(f"  Mean elevation:  {out['elev_mean_m']:.2f} m")
        print(f"  Pixel count:     {out['n_pixels']:,}\n")


Compound (depth 2.5–6.0 m):
  Elevation range: 0.00 – 8.97 m
  Mean elevation:  2.60 m
  Pixel count:     203,115

Bathtub (depth 2.5–6.0 m):
  Elevation range: 1.50 – 5.00 m
  Mean elevation:  2.62 m
  Pixel count:     215,919

